# Notebook 01 — Setup and Data Download

**Mine Presence Classifier Project**

This notebook is the first step in building a classifier that predicts the presence of artisanal mines from satellite imagery. In this notebook we will:

1. Set up our Python environment and authenticate with Google Earth Engine (GEE).
2. Download artisanal mine location data from the IPIS (International Peace Information Service) WFS endpoint.
3. Clean and filter the data to our study area in eastern DRC.
4. Visually inspect the mine locations on an interactive map.
5. Save the cleaned dataset for use in subsequent notebooks.

---

## 1. Setup — Import Libraries and Initialise Google Earth Engine

We begin by importing the libraries we need:

| Library | Purpose |
|---|---|
| `ee` | Google Earth Engine Python API — access satellite imagery and geospatial datasets |
| `geemap` | Interactive mapping built on top of `ee` and `ipyleaflet` |
| `geopandas` | Extends pandas with spatial data types and operations |
| `pandas` | General-purpose data manipulation |
| `owslib` | OGC web-service client — we use it to query WFS endpoints |

After importing, we authenticate with GEE. The first time you run `ee.Authenticate()` it will open a browser window where you grant access. The resulting token is cached locally so you only need to do this once per machine.

In [1]:
# --- Standard library / data ---
import pandas as pd
import geopandas as gpd

# --- Google Earth Engine ---
import ee
import geemap

# --- OGC web services (WFS) ---
from owslib.wfs import WebFeatureService

# --- Misc ---
import json
from pathlib import Path

import folium

---

## 2. Download IPIS Artisanal Mine Locations via WFS

The **International Peace Information Service (IPIS)** publishes curated data on artisanal and small-scale mining sites in the Democratic Republic of the Congo (DRC). The data are available through an OGC-compliant **Web Feature Service (WFS)** endpoint.

We will:
1. Connect to the IPIS GeoServer WFS endpoint.
2. Request the `cod_mines_curated_all_opendata_p_ipis` layer in GeoJSON format.
3. Parse the response into a `GeoDataFrame` for further processing.

> **What is WFS?**  
> WFS is an OGC standard that allows clients to request geospatial *features* (vector data) from a server. Think of it as a spatial API — you send a request and get back points, lines, or polygons with their attributes.

In [2]:
# Authenticate with Google Earth Engine.
# A browser window will open the first time you run this.
ee.Authenticate()

True

In [3]:
# Initialise the Earth Engine API.
# IMPORTANT: Replace 'ee-YOUR_PROJECT_ID' with your own GEE cloud project ID.
ee.Initialize(project='concise-ion-489212-a9')

print('Google Earth Engine initialised successfully.')

Google Earth Engine initialised successfully.


In [4]:
# --- Connect to the IPIS WFS endpoint ---
WFS_URL = 'http://geo.ipisresearch.be/geoserver/wfs'
LAYER_NAME = 'cod_mines_curated_all_opendata_p_ipis'

print(f'Connecting to WFS: {WFS_URL}')
wfs = WebFeatureService(url=WFS_URL, version='2.0.0')

# List available layers (uncomment to explore)
# print(list(wfs.contents.keys()))

print(f'Requesting layer: {LAYER_NAME} ...')

# Request the full layer in GeoJSON format
response = wfs.getfeature(
    typename=[LAYER_NAME],
    outputFormat='application/json'
)

# Parse the JSON response and load into a GeoDataFrame
data = json.loads(response.read())
gdf_mines_raw = gpd.GeoDataFrame.from_features(data['features'], crs='EPSG:4326')

print(f'Downloaded {len(gdf_mines_raw)} mine site records.')
gdf_mines_raw.head()

Connecting to WFS: http://geo.ipisresearch.be/geoserver/wfs
Requesting layer: cod_mines_curated_all_opendata_p_ipis ...
Downloaded 7163 mine site records.


,geometry,vid,cid,id,source,project,pcode,name,visit_date,visit_onsite,...,state_service1,state_service2,state_service3,state_service4,traceability,qualification,childunder15,childunder15work,women,womenwork
0,POINT (28.71526 0.33702),1,1,1,IPIS - Ministère des Mines,IPIS - 2009,codmine00191,Eohe,2009-01-01Z,1,...,None,None,None,None,None,None,NaN,None,NaN,None
1,POINT (28.69916 0.32153),2,2,2,IPIS - Ministère des Mines,IPIS - 2009,codmine00192,Eita,2009-01-01Z,1,...,None,None,None,None,None,None,NaN,None,NaN,None
2,POINT (28.18514 0.54499),3,3,3,IPIS - Ministère des Mines,IPIS - 2009,codmine00242,Mungu Iko,2009-01-01Z,1,...,None,None,None,None,None,None,NaN,None,NaN,None
3,POINT (28.88453 -0.35253),4,4,4,IPIS - Ministère des Mines,IPIS - 2009,codmine00260,Kiviri/Tayna,2009-01-01Z,1,...,None,None,None,None,None,None,NaN,None,NaN,None
4,POINT (28.90394 -0.03671),5,5,5,IPIS - Ministère des Mines,IPIS - 2009,codmine00272,Makanga,2009-01-01Z,1,...,None,None,None,None,None,None,NaN,None,NaN,None


---

## 3. Clean and Filter the Mine Data

Raw geospatial datasets almost always need cleaning. Here we will:

1. **Drop rows with missing coordinates** — some records may lack valid geometry.
2. **Remove duplicate geometries** — the same mine may appear more than once.
3. **Spatial filter** — restrict to our study area in **eastern DRC** (longitude 25°–31° E, latitude 5° S – 3° N).
4. **Add a label column** — we assign `label = 1` to every row, meaning *mine present*. Later notebooks will generate negative samples (`label = 0`) for locations where no mine is known.

In [5]:
# --- 3a. Drop rows with missing coordinates ---
gdf_mines = gdf_mines_raw.copy()
before = len(gdf_mines)
gdf_mines = gdf_mines[gdf_mines.geometry.notnull()]
gdf_mines = gdf_mines[~gdf_mines.is_empty]
print(f'Dropped {before - len(gdf_mines)} rows with missing/empty geometry.')

Dropped 0 rows with missing/empty geometry.


In [6]:
# --- 3b. Remove duplicate geometries ---
before = len(gdf_mines)
gdf_mines = gdf_mines.drop_duplicates(subset='geometry')
print(f'Removed {before - len(gdf_mines)} duplicate geometries.')
print(f'Remaining records: {len(gdf_mines)}')

Removed 3740 duplicate geometries.
Remaining records: 3423


In [7]:
# --- 3c. Filter to the eastern DRC study area ---
# Bounding box: longitude 25-31, latitude -5 to 3
LON_MIN, LON_MAX = 25, 31
LAT_MIN, LAT_MAX = -5, 3

before = len(gdf_mines)
gdf_mines = gdf_mines.cx[LON_MIN:LON_MAX, LAT_MIN:LAT_MAX]
print(f'Filtered to study area ({LON_MIN}°–{LON_MAX}° E, {LAT_MIN}°–{LAT_MAX}° N).')
print(f'Kept {len(gdf_mines)} of {before} records.')

Filtered to study area (25°–31° E, -5°–3° N).
Kept 2842 of 3423 records.


In [8]:
# --- 3d. Add a label column (1 = mine present) and source tag ---
gdf_mines['label'] = 1
gdf_mines['source'] = 'ipis'

print(f'Final cleaned IPIS dataset shape: {gdf_mines.shape}')
gdf_mines.head()

Final cleaned IPIS dataset shape: (2842, 85)


,geometry,vid,cid,id,source,project,pcode,name,visit_date,visit_onsite,...,state_service2,state_service3,state_service4,traceability,qualification,childunder15,childunder15work,women,womenwork,label
0,POINT (28.71526 0.33702),1,1,1,ipis,IPIS - 2009,codmine00191,Eohe,2009-01-01Z,1,...,None,None,None,None,None,NaN,None,NaN,None,1
1,POINT (28.69916 0.32153),2,2,2,ipis,IPIS - 2009,codmine00192,Eita,2009-01-01Z,1,...,None,None,None,None,None,NaN,None,NaN,None,1
2,POINT (28.18514 0.54499),3,3,3,ipis,IPIS - 2009,codmine00242,Mungu Iko,2009-01-01Z,1,...,None,None,None,None,None,NaN,None,NaN,None,1
3,POINT (28.88453 -0.35253),4,4,4,ipis,IPIS - 2009,codmine00260,Kiviri/Tayna,2009-01-01Z,1,...,None,None,None,None,None,NaN,None,NaN,None,1
4,POINT (28.90394 -0.03671),5,5,5,ipis,IPIS - 2009,codmine00272,Makanga,2009-01-01Z,1,...,None,None,None,None,None,NaN,None,NaN,None,1


---

## 4. Visual Inspection — Interactive Map

Before we save the data it is always a good idea to **look at it on a map**. This helps catch obvious errors (e.g., points that fall in the ocean) and gives us intuition about the spatial distribution of mines in the study area.

We use `geemap` to create an interactive Leaflet map centred on eastern DRC and overlay the mine locations as circle markers.

In [9]:
# --- Create an interactive map centred on the study area ---
centre_lat = (LAT_MIN + LAT_MAX) / 2  # -1.0
centre_lon = (LON_MIN + LON_MAX) / 2  # 28.0

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=7)
for _, row in gdf_mines.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color='red',
        fill=True,
    ).add_to(m)
m

---

## 4b. Load Industrial Mine Data

To complement the IPIS artisanal/small-scale mines, we integrate two additional data sources that capture **industrial-scale** mining operations:

1. **Maus et al. (2022) Global Mining Polygons V2** — 44,929 satellite-delineated mining polygons worldwide. We extract centroids of polygons within our study area.
2. **USGS Africa Mineral Industries Geodatabase** — point locations of mineral production/processing facilities in Africa.

> **Download instructions (manual):**
> - Maus polygons: Download GeoPackage from [PANGAEA](https://doi.pangaea.de/10.1594/PANGAEA.942325) → place at `data/raw/maus/global_mining_polygons_v2.gpkg`
> - USGS geodatabase: Download from [ScienceBase](https://www.sciencebase.gov/catalog/item/607611a9d34e018b3201cbbf) → unzip to `data/raw/usgs/Africa_GIS.gdb`

In [10]:
# --- 4b-i. Load & filter Maus Global Mining Polygons ---
maus_path = Path('../data/raw/maus/global_mining_polygons_v2.gpkg')

if maus_path.exists():
    gdf_maus_raw = gpd.read_file(maus_path)
    print(f'Loaded {len(gdf_maus_raw)} Maus mining polygons worldwide.')

    # Filter to study area bounding box
    gdf_maus_polys = gdf_maus_raw.cx[LON_MIN:LON_MAX, LAT_MIN:LAT_MAX].copy()
    print(f'Filtered to study area: {len(gdf_maus_polys)} polygons.')

    # Keep original polygons for dedup — centroids are computed after dedup
    gdf_maus_polys['label'] = 1
    gdf_maus_polys['source'] = 'maus'
    gdf_maus_polys = gdf_maus_polys[['geometry', 'label', 'source']].reset_index(drop=True)
    display(gdf_maus_polys.head())
else:
    gdf_maus_polys = gpd.GeoDataFrame(columns=['geometry', 'label', 'source'], geometry='geometry', crs='EPSG:4326')
    print(f'⚠ Maus GeoPackage not found at {maus_path.resolve()}')
    print('  Download from https://doi.pangaea.de/10.1594/PANGAEA.942325')

Loaded 44929 Maus mining polygons worldwide.
Filtered to study area: 41 polygons.


,geometry,label,source
0,"POLYGON ((27.531 -4.0268, 27.5304 -4.0304, 27....",1,maus
1,"POLYGON ((27.5504 -4.0196, 27.5481 -4.022, 27....",1,maus
2,"POLYGON ((27.5395 -4.0241, 27.5398 -4.0248, 27...",1,maus
3,"POLYGON ((27.5487 -4.0168, 27.5485 -4.0176, 27...",1,maus
4,"POLYGON ((27.5452 -4.008, 27.5465 -4.0089, 27....",1,maus


In [11]:
# --- 4b-ii. Load & filter USGS Africa Mineral Industries Geodatabase ---
usgs_path = Path('../data/raw/usgs/Africa_GIS.gdb')

if usgs_path.exists():
    # List layers to find the right one
    layers = gpd.list_layers(usgs_path)
    print(f'Available layers in USGS geodatabase:\n{layers}\n')

    # Look for a layer containing mineral/mine facility points
    target_layer = None
    for _, row in layers.iterrows():
        name = row['name']
        if 'mineral' in name.lower() or 'mine' in name.lower() or 'facility' in name.lower():
            target_layer = name
            break
    if target_layer is None:
        target_layer = layers.iloc[0]['name']  # fallback to first layer

    print(f'Reading layer: {target_layer}')
    gdf_usgs_raw = gpd.read_file(usgs_path, layer=target_layer)
    print(f'Loaded {len(gdf_usgs_raw)} records from USGS.')

    # Keep only point geometries
    gdf_usgs = gdf_usgs_raw[gdf_usgs_raw.geometry.geom_type == 'Point'].copy()

    # Filter to DRC if a country column exists
    country_cols = [c for c in gdf_usgs.columns if 'country' in c.lower() or 'cntry' in c.lower()]
    if country_cols:
        col = country_cols[0]
        unique_countries = gdf_usgs[col].unique()
        drc_values = [v for v in unique_countries if v and ('congo' in str(v).lower() or 'drc' in str(v).lower() or 'zaire' in str(v).lower())]
        if drc_values:
            gdf_usgs = gdf_usgs[gdf_usgs[col].isin(drc_values)]
            print(f'Filtered to DRC ({drc_values}): {len(gdf_usgs)} records.')

    # Filter to study area bounding box
    gdf_usgs = gdf_usgs.cx[LON_MIN:LON_MAX, LAT_MIN:LAT_MAX]
    print(f'Filtered to study area: {len(gdf_usgs)} records.')

    gdf_usgs['label'] = 1
    gdf_usgs['source'] = 'usgs'
    gdf_usgs = gdf_usgs[['geometry', 'label', 'source']].reset_index(drop=True)

    print(f'USGS mineral facility points in study area: {len(gdf_usgs)}')
    display(gdf_usgs.head())
else:
    gdf_usgs = gpd.GeoDataFrame(columns=['geometry', 'label', 'source'], geometry='geometry', crs='EPSG:4326')
    print(f'⚠ USGS geodatabase not found at {usgs_path.resolve()}')
    print('  Download from https://www.sciencebase.gov/catalog/item/607611a9d34e018b3201cbbf')

Available layers in USGS geodatabase:
                                name      geometry_type
0             AFR_Mineral_Facilities            Point Z
1               AFR_Mineral_Deposits            Point Z
2            AFR_Mineral_Exploration            Point Z
3         AFR_Mineral_Resources_Coal       MultiPolygon
4       AFR_Mineral_Resources_Copper       MultiPolygon
5        AFR_Mineral_Resources_Gabon       MultiPolygon
6   AFR_Mineral_Resources_Mauritania       MultiPolygon
7          AFR_Mineral_Resources_PGE       MultiPolygon
8       AFR_Mineral_Resources_Potash       MultiPolygon
9             AFR_Infra_OG_Pipelines    MultiLineString
10          AFR_Infra_Power_Stations            Point Z
11       AFR_OG_Provinces_Continuous       MultiPolygon
12     AFR_OG_Provinces_Conventional       MultiPolygon
13      AFR_OG_Resources_Recoverable       MultiPolygon
14         AFR_Infra_Transport_Ports            Point Z
15          AFR_Infra_Transport_Rail    MultiLineString
16        

/Users/maxpieter/Documents/GDS/mines/venv/lib/python3.14/site-packages/pyogrio/core.py:129: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
/Users/maxpieter/Documents/GDS/mines/venv/lib/python3.14/site-packages/pyogrio/core.py:129: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
/Users/maxpieter/Documents/GDS/mines/venv/lib/python3.14/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_read(


,geometry,label,source
0,POINT Z (29.14058 -0.98648 0),1,usgs
1,POINT Z (28.90239 -1.71389 0),1,usgs
2,POINT Z (30.17265 1.97438 0),1,usgs
3,POINT Z (26.69827 -3.57096 0),1,usgs
4,POINT Z (26.71876 -2.58018 0),1,usgs


### Merge All Sources and Deduplicate

We combine IPIS, Maus, and USGS points into a single dataset. To avoid double-counting where datasets overlap:

- **Maus vs IPIS**: If any IPIS point falls *inside* a Maus polygon, that polygon is already represented → drop it. This uses the actual mine footprint rather than an arbitrary distance buffer.
- **USGS vs IPIS**: Remove USGS points within **200 m** of an IPIS point (catches true duplicates without merging distinct nearby sites).
- **USGS vs Maus**: Remove USGS points that fall inside a remaining Maus polygon.
- After dedup, Maus polygons are converted to **centroids** for the point-based pipeline.

In [12]:
# --- 4b-iii. Merge all sources and deduplicate ---
from shapely.ops import unary_union

# Prepare IPIS subset with same columns
gdf_ipis = gdf_mines[['geometry', 'label', 'source']].copy().reset_index(drop=True)

print(f'Before merge:')
print(f'  IPIS:  {len(gdf_ipis)} points')
print(f'  Maus:  {len(gdf_maus_polys)} polygons')
print(f'  USGS:  {len(gdf_usgs)} points')

crs_metric = 'EPSG:32735'  # UTM zone 35S — covers eastern DRC

# --- 1. Maus vs IPIS: drop Maus polygons that contain any IPIS point ---
if len(gdf_maus_polys) > 0 and len(gdf_ipis) > 0:
    maus_m = gdf_maus_polys.to_crs(crs_metric)
    ipis_m = gdf_ipis.to_crs(crs_metric)
    # For each Maus polygon, check if any IPIS point is inside it
    ipis_points_union = unary_union(ipis_m.geometry)
    mask = ~maus_m.geometry.intersects(ipis_points_union)
    n_removed = len(gdf_maus_polys) - mask.sum()
    gdf_maus_polys = gdf_maus_polys[mask.values].reset_index(drop=True)
    print(f'  Removed {n_removed} Maus polygons containing IPIS points.')

# --- 2. USGS vs IPIS: drop USGS points within 200 m of any IPIS point ---
if len(gdf_usgs) > 0 and len(gdf_ipis) > 0:
    ipis_m = gdf_ipis.to_crs(crs_metric)
    usgs_m = gdf_usgs.to_crs(crs_metric)
    ipis_buffer = unary_union(ipis_m.geometry.buffer(200))
    mask = ~usgs_m.geometry.within(ipis_buffer)
    n_removed = len(gdf_usgs) - mask.sum()
    gdf_usgs = gdf_usgs[mask.values].reset_index(drop=True)
    print(f'  Removed {n_removed} USGS points within 200 m of IPIS points.')

# --- 3. USGS vs Maus: drop USGS points inside remaining Maus polygons ---
if len(gdf_usgs) > 0 and len(gdf_maus_polys) > 0:
    maus_m = gdf_maus_polys.to_crs(crs_metric)
    usgs_m = gdf_usgs.to_crs(crs_metric)
    maus_union = unary_union(maus_m.geometry)
    mask = ~usgs_m.geometry.within(maus_union)
    n_removed = len(gdf_usgs) - mask.sum()
    gdf_usgs = gdf_usgs[mask.values].reset_index(drop=True)
    print(f'  Removed {n_removed} USGS points inside Maus polygons.')

# --- 4. Convert remaining Maus polygons to centroids ---
gdf_maus = gdf_maus_polys.copy()
gdf_maus['geometry'] = gdf_maus.to_crs(crs_metric).centroid.to_crs('EPSG:4326')
print(f'  Converted {len(gdf_maus)} Maus polygons to centroids.')

# --- 5. Concatenate all sources ---
gdf_combined = pd.concat([gdf_ipis, gdf_maus, gdf_usgs], ignore_index=True)
gdf_combined = gpd.GeoDataFrame(gdf_combined, geometry='geometry', crs='EPSG:4326')

print(f'\nAfter merge and deduplication:')
print(f'  Total:  {len(gdf_combined)} points')
print(gdf_combined['source'].value_counts().to_string())
print(f'\n  New industrial mines added: {len(gdf_combined) - len(gdf_ipis)}')

Before merge:
  IPIS:  2842 points
  Maus:  41 polygons
  USGS:  17 points
  Removed 4 Maus polygons containing IPIS points.
  Removed 7 USGS points within 200 m of IPIS points.
  Removed 1 USGS points inside Maus polygons.
  Converted 37 Maus polygons to centroids.

After merge and deduplication:
  Total:  2888 points
source
ipis    2842
maus      37
usgs       9

  New industrial mines added: 46


---

## 5. Save the Cleaned Data

We save the cleaned `GeoDataFrame` as a **GeoJSON** file so that subsequent notebooks can load it directly without repeating the download and cleaning steps.

The file is written to `../data/raw/ipis/ipis_mines_cleaned.geojson` (relative to this notebook).

In [13]:
# --- Ensure the output directory exists ---
output_dir = Path('../data/raw/ipis')
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'ipis_mines_cleaned.geojson'

# Save the combined dataset (IPIS + Maus + USGS) to preserve the downstream pipeline contract
gdf_combined.to_file(output_path, driver='GeoJSON')

print(f'Saved {len(gdf_combined)} mine records to {output_path.resolve()}')
print(f'  Sources: {gdf_combined["source"].value_counts().to_dict()}')

Saved 2888 mine records to /Users/maxpieter/Documents/GDS/mines/data/raw/ipis/ipis_mines_cleaned.geojson
  Sources: {'ipis': 2842, 'maus': 37, 'usgs': 9}


---

## Summary

In this notebook we:

- Authenticated and initialised Google Earth Engine.
- Downloaded **artisanal mine** locations from the IPIS WFS endpoint.
- Cleaned the IPIS data by removing missing geometries, deduplicating, and filtering to the eastern DRC study area.
- Loaded **industrial mine** data from two additional sources:
  - **Maus et al. (2022)** Global Mining Polygons (centroids of polygons in our study area)
  - **USGS Africa Mineral Industries Geodatabase** (point locations filtered to DRC)
- Merged all three sources with smart deduplication:
  - Maus polygons containing an IPIS point are dropped (spatial containment)
  - USGS points within 200 m of IPIS are dropped
  - Remaining Maus polygons converted to centroids
- Labelled all records as `label = 1` (mine present) with a `source` column for provenance tracking.
- Saved the combined dataset as a GeoJSON file.

**Next notebook:** We will generate *negative* sample locations (areas without known mines) and combine them with these positive samples to create a balanced training dataset.